# Verify Converted TRELLIS.2 Mapper Checkpoints

This notebook redefines the verification logic locally. It compares each converted `Swin3DLatentMapper` checkpoint against the corresponding legacy `WindowGridFeatMapper` checkpoint on random sparse coordinates.

In [1]:
import os
from contextlib import nullcontext
from dataclasses import replace
from pathlib import Path

import sys
import torch

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "symtrellis").exists():
    REPO_ROOT = REPO_ROOT.parent
OLD_REPO_ROOT = Path(os.environ["SYMTRELLIS_LEGACY_REPO_ROOT"])

sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(OLD_REPO_ROOT))

from src.models.window_grid_feat_mapper import WindowGridFeatMapper, window_grid_feat_mapper_configs
from symtrellis.mapper import Swin3DLatentMapper, Swin3DLatentMapperConfig

In [2]:
DEVICE = "cuda:0"
SEED = 20260626
BACKENDS = ("xformers", "flash_attn")

CKPT_SPECS = [
    {
        "name": "sparse_structure",
        "old_path": Path(os.environ["SYMTRELLIS_LEGACY_SS_MAPPER_CKPT"]),
        "new_path": REPO_ROOT / "checkpoints" / "trellis2_sparse_structure_swin3d_latent_mapper_base.pt",
    },
    {
        "name": "shape_latent",
        "old_path": Path(os.environ["SYMTRELLIS_LEGACY_SHAPE_MAPPER_CKPT"]),
        "new_path": REPO_ROOT / "checkpoints" / "trellis2_shape_latent_swin3d_latent_mapper_small.pt",
    },
]

In [3]:
verification_results = []

torch.set_grad_enabled(False)

for spec in CKPT_SPECS:
    old_ckpt = torch.load(spec["old_path"], map_location="cpu")
    new_ckpt = torch.load(spec["new_path"], map_location="cpu")
    old_train_cfg = old_ckpt["config"]

    grid_size = old_train_cfg["grid_size"]
    feat_dim = old_train_cfg["feat_dim"]
    low = max(0, grid_size // 2 - 4)
    high = min(grid_size, grid_size // 2 + 4)

    grid = torch.stack(
        torch.meshgrid(
            torch.arange(low, high),
            torch.arange(low, high),
            torch.arange(low, high),
            indexing="ij",
        ),
        dim=-1,
    ).reshape(-1, 3)

    generator = torch.Generator(device="cpu")
    generator.manual_seed(SEED + grid_size + feat_dim)
    grid = grid[torch.randperm(grid.shape[0], generator=generator)[:128]]

    relation_ids = torch.arange(2).repeat_interleave(grid.shape[0])[:, None]
    grid_repeated = grid.repeat(2, 1)
    coords_src_cpu = torch.cat([relation_ids, grid_repeated], dim=1).to(dtype=torch.int32)
    coords_dst_cpu = coords_src_cpu.clone()

    O_cpu = torch.eye(3).repeat(2, 1, 1)
    O_cpu[1] = torch.tensor(
        [
            [0.0, -1.0, 0.0],
            [1.0, 0.0, 0.0],
            [0.0, 0.0, 1.0],
        ]
    )
    center_cpu = torch.tensor([(grid_size - 1) / 2, (grid_size - 1) / 2, (grid_size - 1) / 2])
    t_cpu = torch.zeros(2, 3)
    t_cpu[1] = center_cpu - center_cpu @ O_cpu[1].T
    s_cpu = torch.ones(2, dtype=torch.int64)

    for backend in BACKENDS:
        autocast_context = torch.autocast(device_type="cuda", dtype=torch.float16) if backend == "flash_attn" else nullcontext()

        old_model_cfg = window_grid_feat_mapper_configs(
            feat_dim=old_train_cfg["feat_dim"],
            grid_size=old_train_cfg["grid_size"],
            lowrank_rank=old_train_cfg["lowrank_rank"],
        )[old_train_cfg["model_scale"]]
        old_model_cfg = replace(old_model_cfg, attn_backend=backend)
        old_model = WindowGridFeatMapper(old_model_cfg)
        old_model.load_state_dict(old_ckpt["model"], strict=True)
        old_model = old_model.eval().to(device=DEVICE, dtype=torch.float32)

        new_cfg_dict = dict(new_ckpt["config"])
        new_cfg_dict["attn_backend"] = backend
        new_model = Swin3DLatentMapper(Swin3DLatentMapperConfig(**new_cfg_dict))
        new_model.load_state_dict(new_ckpt["model"], strict=True)
        new_model = new_model.eval().to(device=DEVICE, dtype=torch.float32)

        coords_src = coords_src_cpu.to(DEVICE)
        coords_dst = coords_dst_cpu.to(DEVICE)
        O = O_cpu.to(device=DEVICE, dtype=torch.float32)
        t = t_cpu.to(device=DEVICE, dtype=torch.float32)
        s = s_cpu.to(DEVICE)

        with autocast_context:
            old_coeff = old_model(
                a_coords=coords_src,
                b_coords=coords_dst,
                R_b2a=O,
                t_b2a=t,
                s_b2a=s,
            )
            new_coeff = new_model(
                coords_src=coords_src,
                coords_dst=coords_dst,
                O_dst2src=O,
                t_dst2src=t,
                s_dst2src=s,
            )

        old_coeff = old_coeff.to(device=torch.device(DEVICE), dtype=torch.float32)
        new_coeff = new_coeff.to(device=torch.device(DEVICE), dtype=torch.float32)

        feat_generator = torch.Generator(device=DEVICE)
        feat_generator.manual_seed(SEED)
        coeff_dtype = old_coeff.dtype
        feats = torch.randn(coords_src.shape[0], feat_dim, device=DEVICE, dtype=coeff_dtype, generator=feat_generator)
        old_out = old_coeff.apply(feats)
        new_out = new_coeff.apply(feats)

        old_edge_key = old_coeff.e_qids * old_coeff.Na + old_coeff.e_kids
        new_edge_key = new_coeff.e_ids_dst * new_coeff.num_src + new_coeff.e_ids_src
        old_edge_order = torch.argsort(old_edge_key)
        new_edge_order = torch.argsort(new_edge_key)
        same_edge_pairs = torch.equal(old_edge_key[old_edge_order].cpu(), new_edge_key[new_edge_order].cpu())

        verification_results.append(
            {
                "name": spec["name"],
                "backend": backend,
                "dtype": str(coeff_dtype),
                "num_edges_old": int(old_coeff.e_qids.numel()),
                "num_edges_new": int(new_coeff.e_ids_dst.numel()),
                "same_e_dst": bool(torch.equal(old_coeff.e_qids.cpu(), new_coeff.e_ids_dst.cpu())),
                "same_e_src_raw_order": bool(torch.equal(old_coeff.e_kids.cpu(), new_coeff.e_ids_src.cpu())),
                "same_edge_pairs_sorted": bool(same_edge_pairs),
                "max_abs_s_by_edge": float((old_coeff.s[old_edge_order] - new_coeff.s[new_edge_order]).abs().max().item()),
                "max_abs_w_by_edge": float((old_coeff.w[old_edge_order] - new_coeff.w[new_edge_order]).abs().max().item()),
                "max_abs_Ut": float((old_coeff.Ut - new_coeff.Ut).abs().max().item()),
                "max_abs_V": float((old_coeff.V - new_coeff.V).abs().max().item()),
                "max_abs_apply": float((old_out - new_out).abs().max().item()),
            }
        )

        old_model.cpu()
        new_model.cpu()
        torch.cuda.empty_cache()

verification_results

[{'name': 'sparse_structure',
  'backend': 'xformers',
  'dtype': 'torch.float32',
  'num_edges_old': 1807,
  'num_edges_new': 1807,
  'same_e_dst': True,
  'same_e_src_raw_order': False,
  'same_edge_pairs_sorted': True,
  'max_abs_s_by_edge': 0.0,
  'max_abs_w_by_edge': 1.7881393432617188e-07,
  'max_abs_Ut': 0.0,
  'max_abs_V': 0.0,
  'max_abs_apply': 9.5367431640625e-07},
 {'name': 'sparse_structure',
  'backend': 'flash_attn',
  'dtype': 'torch.float32',
  'num_edges_old': 1807,
  'num_edges_new': 1807,
  'same_e_dst': True,
  'same_e_src_raw_order': False,
  'same_edge_pairs_sorted': True,
  'max_abs_s_by_edge': 0.0,
  'max_abs_w_by_edge': 1.1920928955078125e-07,
  'max_abs_Ut': 0.0,
  'max_abs_V': 0.0,
  'max_abs_apply': 1.430511474609375e-06},
 {'name': 'shape_latent',
  'backend': 'xformers',
  'dtype': 'torch.float32',
  'num_edges_old': 1808,
  'num_edges_new': 1808,
  'same_e_dst': True,
  'same_e_src_raw_order': False,
  'same_edge_pairs_sorted': True,
  'max_abs_s_by_edge